# 01 - Data and classical baselines (week 1)
Owner: **Haider**

Fashion-MNIST, T-shirt/top (label $-1$) vs. Shirt (label $+1$), PCA-reduced to the same number of features as the quantum circuits use qubits.

Baselines: majority class, logistic regression, and SVMs with tuned RBF, polynomial and linear kernels. All are tuned by inner cross-validation on training data only, on the same folds as the quantum kernels; the test set is used once.

The baseline with the highest inner-CV accuracy is $C^\ast$, the comparator in our falsifier.

In [1]:
import sys, json
sys.path.insert(0, "..")
import numpy as np
from src.common import get_split, make_preprocessor, inner_folds, result_path
from src.baselines import run_baselines, best_classical, label_shuffle_check

## 1. Settings (same as notebook 02)

In [2]:
SEED     = 0
N_QUBITS = 4          # PCA features, matched to the quantum circuits

## 2. Data

In [3]:
Xtr, ytr, Xte, yte = get_split(SEED)
print("train", Xtr.shape, "test", Xte.shape)
print("train balance", np.bincount((ytr + 1) // 2), "| test balance", np.bincount((yte + 1) // 2))

tf = make_preprocessor(N_QUBITS, alpha=1.0)(Xtr)   # PCA + [-1, 1] scaling, fitted on train only
Ftr, Fte = tf(Xtr), tf(Xte)
folds = inner_folds(ytr, SEED)                     # same folds as notebook 02
print("features:", Ftr.shape[1])

train (80, 784) test (40, 784)
train balance [40 40] | test balance [20 20]
features: 4


## 3. Baseline table

In [4]:
res = run_baselines(Ftr, ytr, Fte, yte, folds)
print(f"{'model':<26}{'inner CV':>9}{'test acc':>9}{'test AUC':>9}   params")
print("-" * 80)
for name, r in res.items():
    cv = "   -" if r["inner_cv"] is None else f"{r['inner_cv']:.3f}"
    print(f"{name:<26}{cv:>9}{r['acc']:>9.3f}{r['auc']:>9.3f}   {r['params']}")
c_star = best_classical(res)
print(f"\nC* (highest inner-CV SVM): {c_star}")

model                      inner CV test acc test AUC   params
--------------------------------------------------------------------------------
Majority class                    -    0.500    0.500   {}
Logistic regression           0.762    0.875    0.887   {'C': 0.001}
SVM RBF (tuned)               0.825    0.850    0.917   {'C': 100, 'gamma': 0.1}
SVM polynomial (tuned)        0.812    0.825    0.923   {'C': 100, 'coef0': 1, 'degree': 3, 'gamma': 0.1}
SVM linear (tuned)            0.762    0.875    0.887   {'C': 0.001}

C* (highest inner-CV SVM): SVM RBF (tuned)


**Read the majority-class row first** (NB11). It is the floor every model must clear.

## 4. Leakage check (NB11): train on shuffled labels, accuracy must fall to chance

In [5]:
acc_shuf = label_shuffle_check(Ftr, ytr, Fte, yte, folds, seed=SEED)
chance = max(np.bincount((yte + 1) // 2)) / len(yte)
print(f"shuffled-label test accuracy: {acc_shuf:.3f}  (chance {chance:.3f})")
print("PASS - no leakage detected" if acc_shuf < chance + 0.15 else "CHECK - investigate preprocessing")

shuffled-label test accuracy: 0.500  (chance 0.500)
PASS - no leakage detected


## 5. Save

In [6]:
out = dict(seed=SEED, n_qubits=N_QUBITS, results=res, c_star=c_star,
           shuffled_label_acc=acc_shuf)
path = result_path("tables", "baselines_v1", SEED, N_QUBITS, "json")
path.write_text(json.dumps(out, indent=2, default=str))
print("saved", path)

saved E:\LUMS\Semester 3\Quantum Machine Learning\Project\AI631-Projected-Quantum-Kernels\results\tables\baselines_v1_seed0_q4.json
